# 4.2 — Subspace and span tests

**Pre-registration (written 2026-10-26, before any score matrix was computed).** Spec: `persona_geom_mixture.md`, Experiment 2.

Question: is the answer-side ("φ-side") subspace of the context × answer score matrix the same for the base model and
the post-trained checkpoints, and does Instruct's own conditional lie in the linear span of base persona rows?

Setup: X_M[s, j] = total log P_M(answer j | header s, question j) in the raw transcript format, for
M ∈ {OLMo 3 base, Instruct-SFT, Instruct-DPO, Instruct, Qwen2.5-7B base}. Rows: 87 contexts (86 personas + generic) plus
about 400–550 more descriptions elicited from the base model with the existing procedure and filters
(`data/prompts/personas_elicited_p4/`, deduplicated against the original 80). Columns: 1,000 base-assistant and 1,000
Instruct answers to training-half questions of `data/basis_split.json` (seed 0).

Analysis choices fixed in advance:
- Principal angles between the top-r right singular vectors, r ∈ {5, 10, 20, 50, 100}, on both the raw matrix and the
  double-centered matrix (row and column means removed); both reported (agreed with the author).
- References: noise ceiling = same model, rows split randomly in half, 10 splits; different family = base vs Qwen;
  random = two random r-dimensional subspaces of R^2000 (analytic expectation and 10 draws).
- Summary number per pair and r: mean squared cosine of the principal angles (= ‖U_aᵀU_b‖²_F / r).
- Power-law exponent α of the singular values, fitted as in Golowich, Liu & Shetty (arXiv:2510.24966, Appendix A.1.1):
  weighted least squares of log σ_i on log(i/n) and log((n−i)/n), weights 1/i, largest singular value excluded.
- Span regression (2b): target y = log P_M(answer | question) in the chat template, for M = Instruct (then SFT, DPO);
  ridge regression onto the base rows plus an intercept, columns split by question into halves (seed 0); ridge penalty
  chosen by 5-fold cross-validation over questions inside the fitting half; report held-out R² and residual RMS in nats
  per answer, with a question bootstrap. Ceiling control: predict the generic row of X_base from the other base rows.

How each outcome will be read (from the spec):

| Result | Reading |
|---|---|
| Base–Instruct overlap ≈ noise ceiling | φ preserved; the log-linear selection assumption holds at the sequence level |
| Base–Instruct overlap drops toward base–Qwen | Post-training rewrote φ about as much as switching model family |
| In between | Report where on the scale, per r; no single reading |

If the error band of a curve overlaps the boundary between two rows, I will say so rather than pick one.

My own prior (for the record): at small r (5–10) the base–Instruct overlap is close to the noise ceiling, because the
leading directions are answer length and generic fluency, which any model shares (and which Qwen will share too, so
base–Qwen will also be high at small r). The informative comparison is at r = 20–50 after double-centering, where I expect
base–Instruct to sit between the ceiling and base–Qwen. For 2b I expect a high R² (above 0.9) for the Instruct row, because
sequence totals are dominated by length and question difficulty, and that this will not distinguish the readings. The
residual RMS relative to the generic-row ceiling is the more informative number.
